<div style="background:#171615;color:#F3F1EC;padding:34px 38px;border-radius:6px;font-family:Verdana,sans-serif">
<div style="color:#FF6B63;font-size:12px;font-weight:700;letter-spacing:.16em">FATEC COTIA · CIÊNCIA DE DADOS · PARADIGMAS E TECNOLOGIAS EMERGENTES EM CD</div>
<div style="font-size:34px;font-weight:700;line-height:1.15;margin:14px 0 10px">Achados e perdidos da Fatec com o foundation model <span style="color:#FF6B63">BLIP</span></div>
<div style="font-size:16px;color:#A9A59F;line-height:1.5">Tire uma foto do objeto: o BLIP descreve e cadastra sozinho. Descreva o que perdeu, em português: o BLIP compara o texto com todas as fotos e mostra as mais parecidas.</div>
</div>

### Por que este caso
O BLIP foi treinado com fotos do dia a dia (COCO, Visual Genome, web). Garrafa, mochila, estojo, fone e caderno são exatamente o tipo de imagem que ele conhece — e a turma pode testar ao vivo, com o celular, pelo QR code.

### As três capacidades do BLIP, usadas de verdade
| Checkpoint | Modo da arquitetura | Papel no projeto |
|---|---|---|
| `Salesforce/blip-image-captioning-base` | decoder (LM) | descrição automática do objeto |
| `Salesforce/blip-vqa-base` | encoder com imagem + decoder | o que é, cor, marca |
| `Salesforce/blip-itm-base-coco` | ITC + ITM | **busca por texto**: nota de quanto cada foto combina com a frase |

> ⚙️ `Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4`, depois **Executar tudo**.

## 1 · Ambiente

In [ ]:
!pip install -q "gradio>=5,<6" transformers pillow matplotlib pandas "qrcode[pil]" deep-translator

import io, json, time, base64, datetime, os, threading, textwrap, uuid
from functools import lru_cache
import numpy as np, pandas as pd, requests, torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, HTML
from transformers import (BlipProcessor, BlipForConditionalGeneration,
                          BlipForQuestionAnswering, BlipForImageTextRetrieval)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
GPU    = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (funciona, mais devagar)"
RED, INK, G6, PAPER = "#B20000", "#151515", "#5F5F5F", "#FBFAF7"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})
display(HTML(f"<div style='font-family:Verdana;border-left:4px solid {RED};padding:6px 14px'><b>Dispositivo:</b> {DEVICE} · <b>GPU:</b> {GPU} · <b>PyTorch</b> {torch.__version__}</div>"))

## 2 · Os três checkpoints do BLIP

In [ ]:
CAP_ID, VQA_ID, ITM_ID = ("Salesforce/blip-image-captioning-base",
                          "Salesforce/blip-vqa-base", "Salesforce/blip-itm-base-coco")
t0 = time.time()
cap_proc = BlipProcessor.from_pretrained(CAP_ID)
cap_model = BlipForConditionalGeneration.from_pretrained(CAP_ID, torch_dtype=DTYPE).to(DEVICE).eval()
vqa_proc = BlipProcessor.from_pretrained(VQA_ID)
vqa_model = BlipForQuestionAnswering.from_pretrained(VQA_ID, torch_dtype=DTYPE).to(DEVICE).eval()
itm_proc = BlipProcessor.from_pretrained(ITM_ID)
itm_model = BlipForImageTextRetrieval.from_pretrained(ITM_ID, torch_dtype=DTYPE).to(DEVICE).eval()
T_CARGA = time.time() - t0
print(f"✅ 3 modelos carregados em {T_CARGA:.1f} s")

### 2.1 Raio-X — parâmetros medidos nos pesos (use estes números no slide)

In [ ]:
def raio_x(m):
    partes = {"ViT (imagem)": m.vision_model}
    if hasattr(m, "text_encoder"): partes["encoder de texto"] = m.text_encoder
    if hasattr(m, "text_decoder"): partes["decoder de texto"] = m.text_decoder
    d = {k: round(sum(p.numel() for p in v.parameters()) / 1e6, 1) for k, v in partes.items()}
    d["TOTAL (M)"] = round(sum(p.numel() for p in m.parameters()) / 1e6, 1)
    d["memória (MB)"] = round(sum(p.numel() * p.element_size() for p in m.parameters()) / 1024**2)
    return d
RAIO_X = pd.DataFrame({"captioning-base": raio_x(cap_model), "vqa-base": raio_x(vqa_model), "itm-base-coco": raio_x(itm_model)}).T.fillna("—")
display(RAIO_X)
if DEVICE == "cuda": print(f"VRAM usada: {torch.cuda.memory_allocated()/1024**2:,.0f} MB")

## 3 · Português ↔ inglês
O tokenizador do BLIP é em inglês. A turma vai digitar em português, então a busca passa por uma tradução (Google Translate via `deep-translator`, um serviço auxiliar — **não** é o foundation model). Se a internet falhar, um dicionário básico assume.

In [ ]:
try:
    from deep_translator import GoogleTranslator
    _pt2en, _en2pt = GoogleTranslator(source="auto", target="en"), GoogleTranslator(source="en", target="pt")
except Exception:
    _pt2en = _en2pt = None

DIC = {"garrafa": "bottle", "mochila": "backpack", "bolsa": "bag", "estojo": "pencil case", "caderno": "notebook",
       "livro": "book", "celular": "cell phone", "fone": "headphones", "chave": "key", "chaves": "keys",
       "carteira": "wallet", "óculos": "glasses", "caneta": "pen", "copo": "cup", "garrafinha": "bottle",
       "carregador": "charger", "notebook": "laptop", "boné": "cap", "blusa": "sweater", "casaco": "jacket",
       "azul": "blue", "preto": "black", "preta": "black", "branco": "white", "branca": "white",
       "vermelho": "red", "vermelha": "red", "verde": "green", "amarelo": "yellow", "amarela": "yellow",
       "rosa": "pink", "roxo": "purple", "roxa": "purple", "cinza": "gray", "marrom": "brown", "laranja": "orange",
       "com": "with", "de": "of", "e": "and"}
DIC_EN = {v: k for k, v in DIC.items() if k not in ("garrafinha", "preta", "branca", "vermelha", "amarela", "roxa")}

@lru_cache(1024)
def para_en(t):
    t = (t or "").strip()
    if not t: return t
    try:
        if _pt2en: return _pt2en.translate(t) or t
    except Exception: pass
    return " ".join(DIC.get(w.lower(), w) for w in t.split())

@lru_cache(1024)
def para_pt(t):
    t = (t or "").strip()
    if not t: return t
    try:
        if _en2pt: return _en2pt.translate(t) or t
    except Exception: pass
    return " ".join(DIC_EN.get(w.lower(), w) for w in t.split())

print(para_en("garrafa azul com adesivo"), "|", para_pt("a black backpack on a chair"))

## 4 · Motor
### 4.1 Cadastro: foto → descrição + atributos

In [ ]:
def carregar_imagem(origem, lado_max=1024):
    if isinstance(origem, Image.Image): img = origem
    elif str(origem).startswith("http"):
        r = requests.get(origem, timeout=30, headers={"User-Agent": "Mozilla/5.0 (Fatec Cotia demo)"}); r.raise_for_status()
        img = Image.open(io.BytesIO(r.content))
    else: img = Image.open(origem)
    img = ImageOps.exif_transpose(img).convert("RGB"); img.thumbnail((lado_max, lado_max))
    return img

@torch.no_grad()
def legenda(img, prompt="a photo of"):
    x = (cap_proc(img, prompt, return_tensors="pt") if prompt else cap_proc(img, return_tensors="pt")).to(DEVICE, DTYPE)
    out = cap_model.generate(**x, max_new_tokens=30, num_beams=4, repetition_penalty=1.3)
    return cap_proc.decode(out[0], skip_special_tokens=True).strip()

@torch.no_grad()
def perguntar(img, q):
    x = vqa_proc(img, q, return_tensors="pt").to(DEVICE, DTYPE)
    out = vqa_model.generate(**x, max_new_tokens=10)
    return vqa_proc.decode(out[0], skip_special_tokens=True).strip()

ATRIBUTOS = {"objeto": "What is the main object in this photo?",
             "cor": "What color is it?",
             "marca": "What brand is it?",
             "onde": "Where is it?"}

ITENS, LOCK = [], threading.Lock()

def thumb_b64(img, lado=360):
    t = img.copy(); t.thumbnail((lado, lado)); b = io.BytesIO(); t.save(b, "JPEG", quality=82)
    return base64.b64encode(b.getvalue()).decode()

@torch.no_grad()
def cadastrar(origem, quem="", local=""):
    t0 = time.time(); img = carregar_imagem(origem)
    leg = legenda(img)
    att = {k: perguntar(img, q) for k, q in ATRIBUTOS.items()}
    pix = itm_proc(images=img, return_tensors="pt")["pixel_values"].cpu()   # guardado para a busca
    item = dict(id=uuid.uuid4().hex[:6].upper(), hora=datetime.datetime.now().strftime("%H:%M:%S"),
                quem=quem.strip() or "anônimo", local=local.strip(), legenda_en=leg, legenda_pt=para_pt(leg),
                **{k: para_pt(v) for k, v in att.items()}, atributos_en=att, tempo_s=round(time.time() - t0, 2))
    with LOCK: ITENS.append({**item, "_img": img, "_pix": pix})
    return item, img

### 4.2 Busca: frase em português → foto
Dois modos da arquitetura MED trabalham juntos:
- **ITC** (contraste): similaridade de cosseno entre o vetor da frase e o vetor de cada foto — rápido, bom para pré-ordenar;
- **ITM** (*matching*): o encoder lê a frase **olhando** a foto por *cross-attention* e dá P(combina) — mais preciso. É a nota final.

In [ ]:
@torch.no_grad()
def buscar(consulta, k=5, lote=16):
    with LOCK: base = list(ITENS)
    if not base or not consulta.strip(): return [], consulta
    q_en = para_en(consulta)
    tx = itm_proc(text=[q_en], return_tensors="pt")
    ids, am = tx["input_ids"].to(DEVICE), tx["attention_mask"].to(DEVICE)
    itm, itc = [], []
    for i in range(0, len(base), lote):
        pv = torch.cat([b["_pix"] for b in base[i:i+lote]]).to(DEVICE, DTYPE); n = pv.shape[0]
        o = itm_model(input_ids=ids.repeat(n, 1), attention_mask=am.repeat(n, 1), pixel_values=pv, use_itm_head=True)
        itm += torch.softmax(o.itm_score.float(), -1)[:, 1].tolist()
        try:
            o2 = itm_model(input_ids=ids, attention_mask=am, pixel_values=pv, use_itm_head=False)
            itc += o2.itm_score.float().reshape(-1).tolist()
        except Exception:
            itc += [float("nan")] * n
    ordem = np.argsort(itm)[::-1][:k]
    return [dict(item=base[j], p_itm=round(itm[j], 3), cos_itc=round(itc[j], 3)) for j in ordem], q_en

## 5 · Visual: cartão do objeto e ranking da busca

In [ ]:
def cartao(item, img=None):
    im = img if img is not None else item.get("_img")
    foto = f"<img src='data:image/jpeg;base64,{thumb_b64(im)}' style='width:100%;display:block'>" if im is not None else ""
    tags = "".join(f"<span style='display:inline-block;border:1px solid #CFCBC4;padding:3px 9px;margin:3px 4px 0 0;font-size:13px'><b>{k}:</b> {item[k]}</span>"
                   for k in ("objeto", "cor", "marca", "onde"))
    return f"""<div style='font-family:Verdana;display:grid;grid-template-columns:220px 1fr;gap:18px;background:{PAPER};border:1px solid #E3E0D9;padding:14px;max-width:760px'>
      <div>{foto}</div><div>
      <div style='font-size:11px;letter-spacing:.14em;font-weight:700;color:{RED}'>CADASTRADO · #{item['id']} · {item['hora']}</div>
      <div style='font-size:20px;font-weight:700;margin:6px 0 2px'>{item['legenda_pt']}</div>
      <div style='font-size:13px;color:{G6};font-style:italic'>“{item['legenda_en']}”</div>
      <div style='margin-top:8px'>{tags}</div>
      <div style='font-size:12px;color:{G6};margin-top:8px'>por {item['quem']}{' · achado em ' + item['local'] if item['local'] else ''} · {item['tempo_s']} s</div></div></div>"""

def ranking_html(res, consulta, q_en):
    if not res: return "<p style='font-family:Verdana'>Nada cadastrado ainda.</p>"
    linhas = ""
    for i, r in enumerate(res):
        it, p = r["item"], r["p_itm"]
        cor = RED if i == 0 else "#9A968F"
        linhas += f"""<div style='display:grid;grid-template-columns:70px 1fr 64px;gap:12px;align-items:center;margin:8px 0'>
          <img src='data:image/jpeg;base64,{thumb_b64(it["_img"], 140)}' style='width:70px;height:70px;object-fit:cover'>
          <div><div style='font-size:14px;{'font-weight:700' if i == 0 else ''}'>{it['legenda_pt']} <span style='color:{G6};font-size:12px'>· #{it['id']} · {it['quem']}</span></div>
          <div style='height:12px;background:#EEEAE4;margin-top:5px'><div style='height:12px;width:{p*100:.0f}%;background:{cor}'></div></div></div>
          <div style='font-weight:700;text-align:right'>{p:.2f}</div></div>"""
    return f"""<div style='font-family:Verdana;background:{PAPER};border:1px solid #E3E0D9;padding:14px 18px'>
      <div style='font-size:11px;letter-spacing:.14em;font-weight:700;color:{RED}'>BUSCA · P(combina) pelo ITM</div>
      <div style='font-size:20px;font-weight:700'>“{consulta}”</div><div style='font-size:13px;color:{G6}'>enviado ao modelo como: “{q_en}”</div>{linhas}</div>"""

## 6 · Teste no notebook
### 6.1 Cadastre fotos de objetos
Fotografe 4 a 8 objetos diferentes (garrafa, mochila, estojo, fone, caderno, chaves…). Elas também entram no mural do app e no vídeo.

In [ ]:
FOTOS = []
try:
    from google.colab import files
    FOTOS = list(files.upload().keys())
except Exception as e:
    print("Upload indisponível:", e)
for f in FOTOS:
    item, img = cadastrar(f, quem="teste do grupo")
    display(HTML(cartao(item, img)))
print(f"{len(ITENS)} objeto(s) no cadastro")

### 6.2 Busque por texto
Troque as frases por descrições dos **seus** objetos.

In [ ]:
BUSCAS = ["garrafa de água", "mochila preta", "fone de ouvido"]
RESULTADOS = []
for q in BUSCAS:
    res, q_en = buscar(q)
    RESULTADOS.append((q, q_en, res))
    display(HTML(ranking_html(res, q, q_en)))

### 6.3 Avaliação honesta: a busca acerta?
Para cada foto, escreva **como alguém que perdeu aquele objeto o descreveria** (sem olhar a legenda do modelo). A célula mede:
- **top-1**: a foto certa ficou em 1º lugar;
- **top-3**: ficou entre as 3 primeiras;
- **MRR**: média de 1/posição da foto certa (1 = perfeito).

In [ ]:
# índice da foto (ordem de ITENS)  →  descrição de quem perdeu
GABARITO = {
    # 0: "garrafa azul de metal",
    # 1: "mochila preta com chaveiro",
}
def avaliar(gabarito):
    if not gabarito: print("Preencha o GABARITO para medir a busca."); return None
    linhas = []
    for idx, q in gabarito.items():
        res, _ = buscar(q, k=len(ITENS))
        ids = [r["item"]["id"] for r in res]; pos = ids.index(ITENS[idx]["id"]) + 1
        linhas.append(dict(foto=idx, busca=q, posicao=pos, acerto_top1=pos == 1, top3=pos <= 3))
    df = pd.DataFrame(linhas); display(df)
    m = dict(n=len(df), top1=df.acerto_top1.mean(), top3=df.top3.mean(), MRR=(1 / df.posicao).mean())
    print(f"n={m['n']} · top-1 = {m['top1']:.0%} · top-3 = {m['top3']:.0%} · MRR = {m['MRR']:.2f}  (acaso: top-1 ≈ {1/len(ITENS):.0%})")
    return m
METRICAS = avaliar(GABARITO)

## 7 · ⭐ App ao vivo para a turma (Gradio + QR code)
Sobe o app com **link público** e mostra um **QR code** grande. Projete o QR (ou cole o link no slide “Ao vivo” da apresentação): cada colega abre no celular, tira foto de um objeto e ele aparece no mural de todos.

In [ ]:
import gradio as gr, qrcode

CSS = """
.gradio-container{font-family:Verdana,sans-serif!important;max-width:1100px!important}
#topo{background:#171615;color:#F3F1EC;padding:16px 20px;border-radius:4px}
#topo h1{margin:0;font-size:22px;color:#F3F1EC} #topo p{margin:4px 0 0;color:#A9A59F;font-size:13px} #topo b{color:#FF6B63}
button.primary{background:#B20000!important;border-color:#B20000!important}
"""

def ui_cadastrar(img, quem, local):
    if img is None: return "<p style='font-family:Verdana'>Tire ou escolha uma foto.</p>", mural()
    item, im = cadastrar(img, quem or "", local or "")
    return cartao(item, im), mural()

def mural():
    with LOCK: base = list(ITENS)[::-1]
    return [(b["_img"], f"#{b['id']} · {b['legenda_pt']} · {b['quem']}") for b in base]

def contador():
    with LOCK: n = len(ITENS)
    return f"<div style='font-family:Verdana;font-size:15px'><b style='font-size:28px;color:#B20000'>{n}</b> objeto(s) cadastrado(s) pela turma</div>"

def ui_buscar(q):
    res, q_en = buscar(q or "")
    return ranking_html(res, q or "", q_en), [(r["item"]["_img"], f"{r['p_itm']:.2f} · {r['item']['legenda_pt']}") for r in res]

def ui_perguntar(img, q):
    if img is None or not q: return ""
    r = perguntar(carregar_imagem(img), para_en(q))
    return f"<div style='font-family:Verdana;border-left:5px solid #B20000;padding:10px 16px;background:#FBFAF7'><div style='font-size:12px;color:#5F5F5F'>{q}</div><div style='font-size:28px;font-weight:700'>{para_pt(r)}</div><div style='font-size:12px;color:#5F5F5F'>modelo respondeu: “{r}”</div></div>"

with gr.Blocks(theme=gr.themes.Base(primary_hue="red", neutral_hue="stone"), css=CSS, title="Achados e perdidos · BLIP") as app:
    gr.HTML("<div id='topo'><h1>Achados e perdidos da Fatec com <b>BLIP</b></h1><p>Tire foto de um objeto: a IA cadastra. Descreva o que perdeu: a IA encontra.</p></div>")
    with gr.Tab("📸 Cadastrar"):
        c_img = gr.Image(type="pil", sources=["upload", "webcam"], label="Foto do objeto", height=320)
        with gr.Row():
            c_quem = gr.Textbox(label="Seu nome (opcional)"); c_loc = gr.Textbox(label="Onde achou? (opcional)")
        c_btn = gr.Button("Cadastrar objeto", variant="primary", size="lg"); c_out = gr.HTML()
    with gr.Tab("🔎 Procurar"):
        b_txt = gr.Textbox(label="Descreva o que você perdeu (em português)", placeholder="garrafa azul com adesivo")
        gr.Examples([["garrafa de água"], ["mochila preta"], ["fone de ouvido branco"], ["caderno"]], inputs=[b_txt])
        b_btn = gr.Button("Procurar", variant="primary"); b_out = gr.HTML(); b_gal = gr.Gallery(columns=5, height=220)
    with gr.Tab("🗂️ Mural"):
        m_n = gr.HTML(contador()); m_gal = gr.Gallery(value=mural(), columns=4, height=520)
    with gr.Tab("💬 Pergunte à IA"):
        p_img = gr.Image(type="pil", sources=["upload", "webcam"], label="Foto", height=300)
        p_txt = gr.Textbox(label="Pergunta (pode ser em português)", placeholder="De que cor é?")
        p_btn = gr.Button("Perguntar", variant="primary"); p_out = gr.HTML()
    with gr.Tab("🧠 Sobre"):
        gr.HTML("<div style='font-family:Verdana;font-size:14px'>" + RAIO_X.to_html(border=0) +
                f"<p>Li, Li, Xiong &amp; Hoi. <i>BLIP</i>. ICML 2022 · {DEVICE} · {GPU}</p></div>")
    c_btn.click(ui_cadastrar, [c_img, c_quem, c_loc], [c_out, m_gal])
    b_btn.click(ui_buscar, b_txt, [b_out, b_gal]); b_txt.submit(ui_buscar, b_txt, [b_out, b_gal])
    p_btn.click(ui_perguntar, [p_img, p_txt], p_out)
    try:
        tm = gr.Timer(3); tm.tick(lambda: (contador(), mural()), None, [m_n, m_gal])
    except Exception:
        gr.Button("Atualizar mural").click(lambda: (contador(), mural()), None, [m_n, m_gal])

app.queue(default_concurrency_limit=1)
_, LOCAL_URL, SHARE_URL = app.launch(share=True, prevent_thread_lock=True, inline=False)
qr = qrcode.make(SHARE_URL, box_size=12, border=2)
b = io.BytesIO(); qr.save(b, format="PNG")
display(HTML(f"""<div style='font-family:Verdana;display:flex;gap:28px;align-items:center;background:#171615;color:#fff;padding:24px'>
  <img src='data:image/png;base64,{base64.b64encode(b.getvalue()).decode()}' style='width:300px;background:#fff'>
  <div><div style='color:#FF6B63;font-weight:700;letter-spacing:.14em;font-size:12px'>APONTE A CÂMERA</div>
  <div style='font-size:30px;font-weight:700;margin:6px 0'>Teste o achados e perdidos</div>
  <div style='font-size:18px;word-break:break-all'><a style='color:#FF8A84' href='{SHARE_URL}' target='_blank'>{SHARE_URL}</a></div>
  <div style='font-size:13px;color:#A9A59F;margin-top:10px'>Cole este link no slide “Ao vivo” da apresentação · válido enquanto o Colab estiver aberto</div></div></div>"""))

## 8 · 🎬 Vídeo do código rodando (resultados reais desta execução)
Gera `demo_blip.mp4`: cada objeto cadastrado (foto, legenda sendo escrita, atributos) e depois as buscas da seção 6.2 (frase digitada, barras de P(combina) subindo, foto vencedora destacada). Tudo vem de `ITENS` e `RESULTADOS`. Arraste o arquivo no slide “Ao vivo”.

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter

def gerar_video(caminho="demo_blip.mp4", fps=15, seg_item=4.5, seg_busca=6.0, max_itens=6):
    with LOCK: itens = list(ITENS)[:max_itens]
    buscas = [r for r in RESULTADOS if r[2]]
    if not itens: print("Cadastre fotos na seção 6.1 antes."); return
    T0, T1 = 2.5, 2.5
    total = T0 + seg_item * len(itens) + seg_busca * len(buscas) + T1
    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    ease = lambda x: 1 - (1 - min(max(x, 0), 1)) ** 3
    def fundo(cor): fig.add_artist(plt.Rectangle((0, 0), 1, 1, transform=fig.transFigure, color=cor, zorder=-10))
    def tela(titulo, linhas):
        fundo(INK)
        fig.text(.07, .64, "FATEC COTIA · PARADIGMAS E TECNOLOGIAS EMERGENTES EM CD", color="#FF6B63", fontsize=13, fontweight="bold")
        fig.text(.07, .52, titulo, color="white", fontsize=30, fontweight="bold")
        for k, l in enumerate(linhas): fig.text(.07, .42 - k * .06, l, color="#CFCBC4", fontsize=16, family="monospace")
    def desenhar(i):
        fig.clf(); t = i / fps
        if t < T0:
            tela("Achados e perdidos com BLIP", [">>> item = cadastrar(foto)", ">>> buscar('mochila preta')", f"{DEVICE} · {GPU} · resultados reais"]); return
        t -= T0
        if t < seg_item * len(itens):
            k = int(t // seg_item); u = t - k * seg_item; it = itens[k]; fundo(PAPER)
            fig.text(.04, .93, f"CADASTRO {k+1}/{len(itens)}  ·  #{it['id']}", fontsize=14, fontweight="bold", color=RED)
            fig.text(.96, .93, f"processado em {it['tempo_s']} s", fontsize=12, color=G6, ha="right")
            ax = fig.add_axes([.04, .06, .44, .82]); ax.imshow(it["_img"]); ax.axis("off")
            n = int(len(it["legenda_pt"]) * ease((u - .3) / 1.4))
            fig.text(.53, .80, "DESCRIÇÃO (captioning)", fontsize=10, color=RED, fontweight="bold")
            fig.text(.53, .76, "\n".join(textwrap.wrap(it["legenda_pt"][:n] + ("" if n >= len(it["legenda_pt"]) else "▌"), 34)), fontsize=21, fontweight="bold", va="top")
            fig.text(.53, .60, f"“{it['legenda_en']}”", fontsize=12, style="italic", color=G6)
            for j, a in enumerate(("objeto", "cor", "marca", "onde")):
                al = ease((u - 1.8 - j * .35) / .4)
                fig.text(.53, .48 - j * .075, f"{a}:", fontsize=15, color=G6, alpha=al)
                fig.text(.63, .48 - j * .075, it[a], fontsize=17, fontweight="bold", alpha=al)
            fig.text(.53, .12, "perguntas VQA → atributos do cadastro", fontsize=11, color=G6)
            return
        t -= seg_item * len(itens)
        if t < seg_busca * len(buscas):
            k = int(t // seg_busca); u = t - k * seg_busca; q, q_en, res = buscas[k]; fundo(PAPER)
            n = int(len(q) * ease(u / 1.0))
            fig.text(.04, .91, "BUSCA:  " + "“" + q[:n] + ("”" if n >= len(q) else "▌"), fontsize=24, fontweight="bold")
            fig.text(.04, .86, f"enviado ao modelo como “{q_en}” · ITM: P(combina) com cada foto", fontsize=12, color=G6)
            g = ease((u - 1.2) / 1.8); m = len(res[:5])
            for j, r in enumerate(res[:5]):
                y = .70 - j * .145
                a = fig.add_axes([.04, y - .055, .09, .125]); a.imshow(r["item"]["_img"]); a.axis("off")
                b = fig.add_axes([.16, y - .025, .40, .05]); b.axis("off"); b.set_xlim(0, 1)
                b.barh(0, 1, color="#EEEAE4", height=1); b.barh(0, r["p_itm"] * g, color=RED if j == 0 else "#B9B5AE", height=1)
                fig.text(.58, y, f"{r['p_itm']*g:.2f}", fontsize=14, fontweight="bold", va="center")
                fig.text(.16, y + .04, r["item"]["legenda_pt"][:46], fontsize=11, color=INK if j == 0 else G6)
            if u > 3.2:
                al = ease((u - 3.2) / .6); top = res[0]["item"]
                ax = fig.add_axes([.66, .16, .31, .62]); ax.imshow(top["_img"]); ax.axis("off")
                ax.set_title("ENCONTRADO", color=RED, fontweight="bold", loc="left", alpha=al)
                fig.text(.66, .10, f"#{top['id']} · cadastrado por {top['quem']}", fontsize=12, alpha=al)
            return
        resumo = [f"objetos cadastrados: {len(ITENS)}", f"buscas: {len(buscas)}"]
        if METRICAS: resumo.append(f"top-1 = {METRICAS['top1']:.0%} · MRR = {METRICAS['MRR']:.2f} (n={METRICAS['n']})")
        tela("Fim da demonstração", resumo)
    FuncAnimation(fig, desenhar, frames=int(total * fps)).save(caminho, writer=FFMpegWriter(fps=fps, bitrate=2400), dpi=100)
    plt.close(fig); print(f"🎬 {caminho} ({total:.0f} s)"); return caminho

video = gerar_video()
if video:
    from IPython.display import Video; display(Video(video, embed=True, width=800))
    try:
        from google.colab import files; files.download(video)
    except Exception: pass

## 9 · Limitações
| Limitação | Efeito | O que o projeto faz |
|---|---|---|
| Zero-shot | objetos muito parecidos (duas garrafas pretas) podem trocar de posição | mostra o top-5 com a nota, não só o 1º |
| Tokenizador em inglês | a frase em português precisa de tradução | tradução automática + dicionário de reserva |
| Marca e texto pequeno | VQA erra logotipos pequenos | marca é só um atributo auxiliar |
| Sem medida ainda | acurácia depende das fotos reais | seção 6.3 mede top-1, top-3 e MRR |

**Referência:** Li, J.; Li, D.; Xiong, C.; Hoi, S. *BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation.* ICML, 2022.